In [ ]:
%pip install -q gensim nltk beautifulsoup4 scikit-learn pandas


In [ ]:
import copy
import csv
import re
from pathlib import Path

import nltk
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from bs4 import BeautifulSoup
from gensim.models import Word2Vec
from nltk.tokenize import sent_tokenize
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
from torch.utils.data import DataLoader, TensorDataset

inputDir = Path('/kaggle/input/competitions/word2vec-nlp-tutorial')
outputDir = Path('/kaggle/working')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('训练设备：', device)


In [ ]:
trainData = pd.read_csv(inputDir / 'labeledTrainData.tsv.zip', sep='\t', compression='zip')
testData = pd.read_csv(inputDir / 'testData.tsv.zip', sep='\t', compression='zip')
unlabeledData = pd.read_csv(
    inputDir / 'unlabeledTrainData.tsv.zip',
    sep='\t',
    compression='zip',
    engine='python',
    quoting=csv.QUOTE_NONE,
    on_bad_lines='error'
)
print('有标签训练集：', trainData.shape)
print('无标签训练集：', unlabeledData.shape)
print('测试集：', testData.shape)


In [ ]:
nltk.download('punkt')
nltk.download('punkt_tab')


def reviewToSentences(review):
    reviewText = BeautifulSoup(review, 'html.parser').get_text()
    rawSentences = sent_tokenize(reviewText.strip())
    sentences = []
    for sentence in rawSentences:
        letters = re.sub(r'[^a-zA-Z]', ' ', sentence)
        words = letters.lower().split()
        if words:
            sentences.append(words)
    return sentences

sentences = []
allReviews = trainData['review'].tolist() + unlabeledData['review'].tolist()
for review in allReviews:
    sentences.extend(reviewToSentences(review))
print('Word2Vec 训练句数：', len(sentences))


In [ ]:
word2vecModel = Word2Vec(
    sentences=sentences,
    vector_size=300,
    window=10,
    min_count=40,
    workers=4,
    sample=1e-3,
    sg=1,
    seed=42,
    epochs=5
)
print('词表大小：', len(word2vecModel.wv))
print('词向量维度：', word2vecModel.vector_size)


In [ ]:
maxVocab = min(50000, len(word2vecModel.wv.index_to_key))

vocabWords = word2vecModel.wv.index_to_key[:maxVocab]

wordToIndex = {
    word: i + 2
    for i, word in enumerate(vocabWords)
}

vocabSize = len(vocabWords) + 2
embeddingDim = word2vecModel.vector_size

embeddingMatrix = np.zeros(
    (vocabSize, embeddingDim),
    dtype=np.float32
)

embeddingMatrix[1] = np.mean(
    word2vecModel.wv[vocabWords],
    axis=0
)

embeddingMatrix[2:] = word2vecModel.wv[vocabWords]

print("Embedding 矩阵形状：", embeddingMatrix.shape)
print("词汇表大小：", vocabSize)
print("词向量维度：", embeddingDim)


In [ ]:
def tokenizeReview(text):

    text = BeautifulSoup(
        str(text),
        "html.parser"
    ).get_text(separator=" ")

    words = re.findall(r"[a-z]+", text.lower())

    return words
tokenizedTrain = [
    tokenizeReview(review)
    for review in trainData["review"].values
]

reviewLengths = [
    len(words)
    for words in tokenizedTrain
]

print("评论长度统计：")
print(pd.Series(reviewLengths).describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]))


In [ ]:
maxLength = int(np.percentile(reviewLengths, 95))
maxLength = min(max(maxLength, 100), 500)
print('最大序列长度：', maxLength)


In [ ]:
def encodeTokens(tokens, maxLength=None):

    if maxLength is None:
        maxLength = maxLength

    ids = [
        wordToIndex.get(word, 1)
        for word in tokens
    ]

    if len(ids) == 0:
        ids = [1]

    ids = ids[:maxLength]

    length = len(ids)

    ids += [0] * (maxLength - length)

    return ids, length


In [ ]:
allSequences = []
allLengths = []

for tokens in tokenizedTrain:
    ids, length = encodeTokens(tokens)

    allSequences.append(ids)
    allLengths.append(length)

allSequences = np.array(allSequences, dtype=np.int64)
allLengths = np.array(allLengths, dtype=np.int64)

allLabels = trainData["sentiment"].values.astype(np.float32)

print("评论序列矩阵：", allSequences.shape)
print("评论长度矩阵：", allLengths.shape)
print("标签矩阵：", allLabels.shape)


In [ ]:
(
    trainSequences, valSequences,
    trainLengths, valLengths,
    trainLabels, valLabels
) = train_test_split(
    allSequences,
    allLengths,
    allLabels,
    test_size=0.2,
    random_state=42,
    stratify=allLabels
)

print("训练集：", trainSequences.shape)
print("验证集：", valSequences.shape)


In [ ]:
batchSize = 64

trainDataset = TensorDataset(
    torch.tensor(trainSequences, dtype=torch.long),
    torch.tensor(trainLengths, dtype=torch.long),
    torch.tensor(trainLabels, dtype=torch.float32)
)

valDataset = TensorDataset(
    torch.tensor(valSequences, dtype=torch.long),
    torch.tensor(valLengths, dtype=torch.long),
    torch.tensor(valLabels, dtype=torch.float32)
)

trainLoader = DataLoader(
    trainDataset,
    batch_size=batchSize,
    shuffle=True,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

valLoader = DataLoader(
    valDataset,
    batch_size=batchSize,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

print("训练批次数：", len(trainLoader))
print("验证批次数：", len(valLoader))


In [ ]:
tokenizedTest = [
    tokenizeReview(text)
    for text in testData["review"].values
]

print("测试集评论数量：", len(tokenizedTest))
print("第一条评论分词结果：", tokenizedTest[0][:20])
testSequences = []
testLengths = []

for tokens in tokenizedTest:

    ids, length = encodeTokens(tokens)

    testSequences.append(ids)
    testLengths.append(length)

testSequences = np.array(testSequences, dtype=np.int64)
testLengths = np.array(testLengths, dtype=np.int64)

testDataset = TensorDataset(
    torch.tensor(testSequences, dtype=torch.long),
    torch.tensor(testLengths, dtype=torch.long)
)

testLoader = DataLoader(
    testDataset,
    batch_size=64,
    shuffle=False
)

print("X_test：", testSequences.shape)
print("len_test：", testLengths.shape)

print("测试集样本数量：", len(testDataset))
print("测试集 batch 数量：", len(testLoader))


In [ ]:
criterion = nn.BCEWithLogitsLoss()

def evaluateModel(net, loader):

    net.eval()

    totalLoss = 0
    allProbabilities = []
    allLabels = []

    with torch.no_grad():

        for x, lengths, labels in loader:

            x = x.to(device)
            labels = labels.to(device)

            logits = net(x, lengths)

            loss = criterion(logits, labels)

            totalLoss += loss.item() * len(labels)

            probs = torch.sigmoid(logits)

            allProbabilities.extend(
                probs.cpu().numpy()
            )

            allLabels.extend(
                labels.cpu().numpy()
            )

    averageLoss = totalLoss / len(loader.dataset)

    auc = roc_auc_score(
        allLabels,
        allProbabilities
    )

    return averageLoss, auc

def fitModel(
    net,
    trainLoader,
    valLoader,
    epochs=10,
    lr=0.001,
    patience=3
):

    optimizer = torch.optim.AdamW(
        net.parameters(),
        lr=lr,
        weight_decay=1e-4
    )

    bestAuc = -1
    bestState = None
    wait = 0

    for epoch in range(1, epochs + 1):

        net.trainData()

        totalLoss = 0

        for x, lengths, labels in trainLoader:

            x = x.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            logits = net(x, lengths)

            loss = criterion(logits, labels)

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                net.parameters(),
                max_norm=1.0
            )

            optimizer.step()

            totalLoss += loss.item() * len(labels)

        trainLoss = totalLoss / len(trainLoader.dataset)

        valLoss, valAuc = evaluateModel(
            net,
            valLoader
        )

        print(
            f"Epoch {epoch:02d} | "
            f"Train Loss: {trainLoss:.4f} | "
            f"Val Loss: {valLoss:.4f} | "
            f"Val AUC: {valAuc:.4f}"
        )

        if valAuc > bestAuc:

            bestAuc = valAuc

            bestState = copy.deepcopy(
                net.state_dict()
            )

            wait = 0

        else:

            wait += 1

        if wait >= patience:

            print("Early Stopping")
            break

    net.load_state_dict(bestState)

    return net, bestAuc


In [ ]:
def predictTest(net, testLoader):
    net.eval()
    predictions = []
    with torch.no_grad():
        for tokens, lengths in testLoader:
            logits = net(tokens.to(device), lengths)
            probabilities = torch.sigmoid(logits)
            predictions.extend(probabilities.cpu().numpy())
    return np.array(predictions)


def saveSubmission(predictions, filename):
    submission = pd.DataFrame({
        'id': testData['id'].values,
        'sentiment': predictions
    })
    submissionPath = outputDir / filename
    submission.to_csv(submissionPath, index=False)
    print('提交文件：', submissionPath)
    print(submission.head())
    return submission


In [ ]:
class LSTMClassifier(nn.Module):

    def __init__(
        self,
        embeddingMatrix,
        hiddenDim=128,
        freezeEmbeddings=False
    ):
        super().__init__()

        embeddingTensor = torch.tensor(
            embeddingMatrix,
            dtype=torch.float32
        )

        self.embedding = nn.Embedding.from_pretrained(
            embeddingTensor,
            freeze=freezeEmbeddings,
            padding_idx=0
        )

        embeddingDim = embeddingMatrix.shape[1]

        self.lstm = nn.LSTM(
            input_size=embeddingDim,
            hidden_size=hiddenDim,
            num_layers=1,
            batch_first=True,
            bidirectional=True
        )

        self.dropout = nn.Dropout(0.5)

        self.fc = nn.Linear(
            hiddenDim * 2,
            1
        )

    def forward(self, x, lengths):

        x = self.embedding(x)

        x = self.dropout(x)

        packed = pack_padded_sequence(
            x,
            lengths.cpu(),
            batch_first=True,
            enforce_sorted=False
        )

        _, (hidden, cell) = self.lstm(packed)

        forwardHidden = hidden[-2]
        backwardHidden = hidden[-1]

        output = torch.cat(
            [forwardHidden, backwardHidden],
            dim=1
        )

        output = self.dropout(output)

        output = self.fc(output)

        return output.squeeze(1)


In [ ]:
print("开始训练 LSTM")

bilstmModel = LSTMClassifier(
    embeddingMatrix,
    hiddenDim=128
).to(device)

bilstmModel, bilstmBestAuc = fitModel(
    bilstmModel,
    trainLoader,
    valLoader,
    epochs=10,
    lr=0.001,
    patience=3
)

print("LSTM 最佳验证集 AUC：", bilstmBestAuc)

torch.save(
    bilstmModel.state_dict(),
    outputDir / "lstm_best.pth"
)
bilstmPredictions = predictTest(
    bilstmModel,
    testLoader
)

bilstmSubmission = saveSubmission(
    bilstmPredictions,
    "submission_lstm.csv"
)


In [ ]:
class AttentionLSTMClassifier(nn.Module):

    def __init__(
        self,
        embeddingMatrix,
        hiddenDim=128,
        freezeEmbeddings=False
    ):
        super().__init__()

        embeddingTensor = torch.tensor(
            embeddingMatrix,
            dtype=torch.float32
        )

        self.embedding = nn.Embedding.from_pretrained(
            embeddingTensor,
            freeze=freezeEmbeddings,
            padding_idx=0
        )

        embeddingDim = embeddingMatrix.shape[1]

        self.lstm = nn.LSTM(
            input_size=embeddingDim,
            hidden_size=hiddenDim,
            num_layers=1,
            batch_first=True,
            bidirectional=True
        )

        self.dropout = nn.Dropout(0.5)

        self.attention = nn.Sequential(
            nn.Linear(hiddenDim * 2, hiddenDim),
            nn.Tanh(),
            nn.Linear(hiddenDim, 1, bias=False)
        )

        self.fc = nn.Linear(
            hiddenDim * 2,
            1
        )

    def forward(self, x, lengths):

        x = self.dropout(self.embedding(x))

        packed = pack_padded_sequence(
            x,
            lengths.cpu(),
            batch_first=True,
            enforce_sorted=False
        )

        packedOutput, _ = self.lstm(packed)
        output, _ = pad_packed_sequence(
            packedOutput,
            batch_first=True
        )

        scores = self.attention(output).squeeze(-1)
        mask = torch.arange(
            output.size(1), device=output.device
        ).unsqueeze(0) < lengths.to(output.device).unsqueeze(1)
        scores = scores.masked_fill(~mask, float('-inf'))
        weights = torch.softmax(scores, dim=1)

        context = torch.sum(output * weights.unsqueeze(-1), dim=1)
        context = self.dropout(context)

        return self.fc(context).squeeze(1)


In [ ]:
print("开始训练 Attention LSTM")

attentionLstmModel = AttentionLSTMClassifier(
    embeddingMatrix,
    hiddenDim=128
).to(device)

attentionLstmModel, attentionLstmBestAuc = fitModel(
    attentionLstmModel,
    trainLoader,
    valLoader,
    epochs=10,
    lr=0.001,
    patience=3
)

print("Attention LSTM 最佳验证集 AUC：", attentionLstmBestAuc)

torch.save(
    attentionLstmModel.state_dict(),
    outputDir / "attention_lstm_best.pth"
)
attentionLstmPredictions = predictTest(
    attentionLstmModel,
    testLoader
)

attentionLstmSubmission = saveSubmission(
    attentionLstmPredictions,
    "submission_attention_lstm.csv"
)


In [ ]:
class CapsuleLayer(nn.Module):

    def __init__(self, inputDim, numCapsules=8, capsuleDim=16, routingIters=3):
        super().__init__()
        self.numCapsules = numCapsules
        self.capsuleDim = capsuleDim
        self.routingIters = routingIters
        self.projection = nn.Linear(
            inputDim, numCapsules * capsuleDim, bias=False
        )

    def squash(self, x):
        squaredNorm = (x ** 2).sum(dim=-1, keepdim=True)
        scale = squaredNorm / (1 + squaredNorm)
        return scale * x / torch.sqrt(squaredNorm + 1e-8)

    def forward(self, x, mask):
        batch_size, sequenceLength, _ = x.shape
        predictions = self.projection(x).reshape(
            batch_size, sequenceLength, self.numCapsules, self.capsuleDim
        )
        logits = x.new_zeros(batch_size, sequenceLength, self.numCapsules)

        for iteration in range(self.routingIters):
            coupling = torch.softmax(logits, dim=2)
            coupling = coupling * mask.unsqueeze(-1)
            weightedSum = (coupling.unsqueeze(-1) * predictions).sum(dim=1)
            capsules = self.squash(weightedSum)

            if iteration < self.routingIters - 1:
                agreement = (predictions * capsules.unsqueeze(1)).sum(dim=-1)
                logits = logits + agreement

        return capsules

class CapsuleLSTMClassifier(nn.Module):

    def __init__(
        self,
        embeddingMatrix,
        hiddenDim=128,
        numCapsules=8,
        capsuleDim=16,
        routingIters=3,
        freezeEmbeddings=False
    ):
        super().__init__()

        embeddingTensor = torch.tensor(
            embeddingMatrix,
            dtype=torch.float32
        )
        self.embedding = nn.Embedding.from_pretrained(
            embeddingTensor,
            freeze=freezeEmbeddings,
            padding_idx=0
        )

        self.lstm = nn.LSTM(
            input_size=embeddingMatrix.shape[1],
            hidden_size=hiddenDim,
            num_layers=1,
            batch_first=True,
            bidirectional=True
        )
        self.dropout = nn.Dropout(0.5)
        self.capsule = CapsuleLayer(
            hiddenDim * 2, numCapsules, capsuleDim, routingIters
        )
        self.fc = nn.Linear(numCapsules * capsuleDim, 1)

    def forward(self, x, lengths):

        x = self.dropout(self.embedding(x))
        packed = pack_padded_sequence(
            x,
            lengths.cpu(),
            batch_first=True,
            enforce_sorted=False
        )

        packedOutput, _ = self.lstm(packed)
        output, _ = pad_packed_sequence(
            packedOutput,
            batch_first=True
        )

        mask = torch.arange(
            output.size(1), device=output.device
        ).unsqueeze(0) < lengths.to(output.device).unsqueeze(1)
        capsules = self.capsule(output, mask)
        features = self.dropout(capsules.flatten(start_dim=1))

        return self.fc(features).squeeze(1)


In [ ]:
print("开始训练 Capsule LSTM")

capsuleLstmModel = CapsuleLSTMClassifier(
    embeddingMatrix,
    hiddenDim=128
).to(device)

capsuleLstmModel, capsuleLstmBestAuc = fitModel(
    capsuleLstmModel,
    trainLoader,
    valLoader,
    epochs=10,
    lr=0.001,
    patience=3
)

print("Capsule LSTM 最佳验证集 AUC：", capsuleLstmBestAuc)

torch.save(
    capsuleLstmModel.state_dict(),
    outputDir / "capsule_lstm_best.pth"
)
capsuleLstmPredictions = predictTest(
    capsuleLstmModel,
    testLoader
)

capsuleLstmSubmission = saveSubmission(
    capsuleLstmPredictions,
    "submission_capsule_lstm.csv"
)


In [ ]:
results = pd.DataFrame([
    {'model': 'BiLSTM', 'validationAuc': bilstmBestAuc},
    {'model': 'Attention-BiLSTM', 'validationAuc': attentionLstmBestAuc},
    {'model': 'Capsule-BiLSTM', 'validationAuc': capsuleLstmBestAuc}
])
results.sort_values('validationAuc', ascending=False)
